# تطبيق عملي: بناء مسار RAG متكامل مدعوم بتوسيع الاستعلامات (Query Expansion)

أهلا بكم في هذا اللاب العملي الشامل للمحاضرة 50.
في هذا المعمل، سنقوم ببناء خط استرجاع وتوليد متكامل (End-to-End RAG Pipeline) يدمج:
1. قراءة وتقطيع ملف بيانات حقيقي يحتوي على شروحات تقنية لمكتبيتي LangChain و CrewAI.
2. بناء مسترجع ذكي يعتمد على خوارزمية أقصى تنوع نسبي (MMR Retriever).
3. بناء سلسلة توسيع الاستعلامات (Query Expansion Chain) باستخدام النموذج اللغوي لإثراء السؤال قبل إرساله للمسترجع.
4. دمج السلاسل مع بعضها باستخدام تقنية RunnableMap لتوليد إجابات دقيقة وموثقة.

### 1. استيراد المكتبات الأساسية
يلا نستورد كل المكتبات اللي هنحتاجها لبناء الـ Pipeline:

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap, RunnableLambda
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
print("تم استيراد كافة المكتبات بنجاح.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_11812\1342352782.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


تم استيراد كافة المكتبات بنجاح.


### 2. تحميل وتقطيع ملف البيانات (LangChain & CrewAI Dataset)
ملف البيانات المرفق يحتوي على تفاصيل دقيقة عن:
* إدارة الذاكرة في LangChain (Buffer, Summary, Window).
* معمارية وكلاء CrewAI ومستويات الذاكرة الثلاثة (Short-term, Long-term, Entity memory).
يلا نحمل الملف ونقطعه بمقاس مناسب:

In [2]:
dataset_file = "langchain_crewai_dataset.txt"

loader = TextLoader(dataset_file, encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.split_documents(raw_docs)

print(f"تم تحميل الملف بنجاح.")
print(f"إجمالي المقاطع الناتجة (Chunks): {len(chunks)} مقطع.")
print("\nعينة من أول مقطع:")
print(chunks[0].page_content[:180] + "...")

تم تحميل الملف بنجاح.
إجمالي المقاطع الناتجة (Chunks): 23 مقطع.

عينة من أول مقطع:
# Comprehensive Technical Documentation: LangChain and CrewAI Frameworks...


### 3. بناء الفهرس المتجهي (FAISS Vector Store)
هنستخدم نموذج التضمين المحلي المجاني الخفيف sentence-transformers/all-MiniLM-L6-v2 لإنشاء الفهرس:

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

print("تم بناء وتسكين المتجهات في FAISS بنجاح.")
print("إجمالي المتجهات المفهرسة:", vector_store.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء وتسكين المتجهات في FAISS بنجاح.
إجمالي المتجهات المفهرسة: 23


### 4. إعداد مسترجع أقصى تنوع نسبي (MMR Retriever)
الجميل هنا إننا مش هنستخدم البحث العادي، بل هنستخدم مسترجع MMR لجلب أفضل 5 مقاطع متنوعة تغطي كل زوايا الموضوع بدون تكرار:

In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 15, "lambda_mult": 0.5}
)
print("تم إعداد مسترجع MMR بنجاح (k=5).")

تم إعداد مسترجع MMR بنجاح (k=5).


### 5. تصميم سلسلة توسيع الاستعلام (Query Expansion Chain)
هنا نضع لمسة الذكاء: قالب التوجيه يطلب من النموذج اللغوي توسيع السؤال بالمصطلحات التقنية والمترادفات المفيدة بدون أن يجيب على السؤال مباشرة:

In [6]:
expansion_prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Expand the following query to improve document retrieval "
    "by adding relevant synonyms, technical terms and useful context.\n"
    "Original query: {query}\n"
    "Expanded query:"
)

groq_key = os.getenv("GROQ_API_KEY")
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
query_expansion_chain = expansion_prompt | llm | StrOutputParser()
print("تم ربط سلسلة التوسيع بالنموذج اللغوي ChatGroq بنجاح.")


تم ربط سلسلة التوسيع بالنموذج اللغوي ChatGroq بنجاح.


### 6. تجربة سلسلة التوسيع على أسئلة المحاضرة
تعال نجرب نشغل سلسلة التوسيع على استعلامات مقتضبة ونشوف النتيجة الممتعة:

In [11]:
test_questions = [
    "langchain memory",
    "Does CrewAI support memory?",
    "CrewAI agents"
]

for q in test_questions:
    exp = query_expansion_chain.invoke({"query": q})
    print("السؤال الأصلي :", q)
    print("السؤال الموسع :", exp)
    print("-" * 50)

السؤال الأصلي : langchain memory
السؤال الموسع : LangChain memory management conversation history state persistence chatbot context window token limit short-term long-term memory vector store database integration Redis PostgreSQL ChromaDB Milvus Qdrant memory types ConversationBufferMemory ConversationSummaryMemory ConversationBufferWindowMemory EntityMemory VectorStoreRetrieverMemory custom memory implementation LLM context retention session state user interaction history
--------------------------------------------------
السؤال الأصلي : Does CrewAI support memory?
السؤال الموسع : CrewAI memory capabilities persistent state context retention long-term memory short-term memory RAG vector database integration knowledge base storage session history agent state management
--------------------------------------------------
السؤال الأصلي : CrewAI agents
السؤال الموسع : CrewAI multi-agent framework autonomous AI agents role-based collaboration LLM orchestration task delegation Crews Flows se

### 7. ربط المسار المتكامل (Building the Full RAG Pipeline)
دلوقتي هنربط المسار كامل:
1. السؤال يدخل على سلسلة التوسيع عشان نجيب النص الموسع.
2. النص الموسع يروح لـ retriever.invoke عشان يجيب أفضل مقاطع.
3. السؤال الأصلي مع السياق يروحوا لسلسلة التوليد عشان النموذج يجاوب استنادا للسياق فقط:

In [12]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_groq import ChatGroq

# 1. قالب الإجابة وسلسلة المستندات
answer_prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context below:\n"
    "Context:\n{context}\n\n"
    "Question: {input}\n"
    "Answer:"
)

gen_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
doc_chain = create_stuff_documents_chain(gen_llm, answer_prompt)

# 2. مسار استخراج السياق: السؤال -> التوسيع -> المسترجع
context_pipeline = (
    (lambda x: {"query": x["input"]})
    | query_expansion_chain 
    | retriever
)

# 3. دمج المسار بالكامل
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": context_pipeline
    })
    | doc_chain
)

print("تم بناء مسار الـ RAG المتكامل بنجاح.")

تم بناء مسار الـ RAG المتكامل بنجاح.


### 8. تشغيل واختبار الـ Pipeline بالكامل
تعال نسأل الأسئلة الثلاثة ونشوف دقة وشمول الإجابات الناتجة عن السياق الموسع:

In [13]:
questions_to_ask = [
    "What types of memory does LangChain support",
    "Does CrewAI support memory?",
    "Tell me about CrewAI agents"
]

for question in questions_to_ask:
    # استدعاء مباشر ونظيف لمسار الـ RAG
    answer = rag_pipeline.invoke({"input": question})
    
    print("=" * 65)
    print(f"السؤال : {question}")
    print(f"الإجابة:\n{answer}")

print("=" * 65)

السؤال : What types of memory does LangChain support
الإجابة:
Based on the context provided, LangChain supports the following types of memory:

*   **ConversationBufferMemory**: Keeps the raw, unmodified chat history in a memory buffer.
*   **ConversationBufferWindowMemory**: Enforces a strict sliding window of the last K interaction pairs, dropping older messages.
*   **VectorStoreRetrieverMemory**: Embeds user inputs and agent responses into a vector database (such as FAISS or Chroma) for persistent semantic state retrieval.
*   **Entity Memory**: Tracks subject-matter entities, user preferences, and business concepts by extracting key entities from agent outputs and indexing their relationships and properties.
*   **Long-Term Memory**: Utilizes local SQLite databases and persistent embeddings to enable agents to learn from past outcomes over weeks and months.
السؤال : Does CrewAI support memory?
الإجابة:
Yes, CrewAI supports memory. It provides an advanced multi-tiered memory archit

In [14]:
for question in questions_to_ask:
    # 1. طباعة السؤال الموسع
    expanded_q = query_expansion_chain.invoke({"query": question})
    
    # 2. تشغيل الـ Pipeline
    answer = rag_pipeline.invoke({"input": question})
    
    print("=" * 65)
    print(f"📌 السؤال الأصلي : {question}")
    print(f"🔍 السؤال الموسع : {expanded_q}")
    print(f"💡 الإجابة النهائية:\n{answer}\n")
print("=" * 65)

📌 السؤال الأصلي : What types of memory does LangChain support
🔍 السؤال الموسع : LangChain memory types short-term long-term conversation history chat history buffer memory summary memory vector store memory entity memory knowledge graph memory Redis memory PostgreSQL memory MongoDB memory ChromaDB memory Pinecone memory Qdrant memory Weaviate memory Milvus memory FAISS memory HNSW memory LSH memory IVF memory memory management state persistence session management context window token limit sliding window rolling buffer memory retrieval augmented generation RAG vector embeddings semantic search similarity search
💡 الإجابة النهائية:
Based on the context provided, LangChain supports the following types of memory:

*   **ConversationBufferMemory**: Keeps the raw, unmodified chat history in a memory buffer.
*   **ConversationBufferWindowMemory**: Enforces a strict sliding window of the last K interaction pairs, dropping older messages.
*   **VectorStoreRetrieverMemory**: Embeds user inputs 